# TALOS Copilot - Objetivo 1: Detección de diferncias anómalas de inventario

## Baseline Model: Z-score por producto + almacén

- **Objetivo 1**: Identificar productos cuya diferencia entre stock teórico y stock físico sea estadísticamente anómala respecto a su comportamiento histórico, cuantificando el impacto económico en pesos y priorizando los hallazgos por severidad.

- **Idea**: 
    - El z-score te dice cuántas desviaciones estándar se aleja un valor de la media.
    - Para cada combinación única de producto + almacén, calculamos qué tan rara es la diferencia de inventario del cierre actual comparada con su historial. Si la diferencia se aleja demasiado de lo normal, se genera una alerta priorizada por impacto económico.

- Importamos librerías.

In [1]:
# Importacion de librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

- Cargamos datos ya procesados. Estos datos contienen:    
    - Solo cierres en estado terminado, finalizado o aplicado
    - Solo productos activos (producto_baja = 0)
    - Solo almacenes activos (almacen_estatus = 1)
    - Solo registros con costopromedio > 0
    - Solo productos con >= 5 cierres históricos


In [2]:
# Carga de datos filtrados para objetivo 1
# Datos exportados desde MySQL con los filtros aplicados
data_obj1 = pd.read_csv('../data/processed/talos.csv')  # cambia el nombre

# Convertir fecha a datetime para poder ordenar correctamente
data_obj1['inventariomes_fecha'] = pd.to_datetime(data_obj1['inventariomes_fecha'])

print(f'Registros cargados: {data_obj1.shape[0]:,}')
print(f'Columnas: {data_obj1.shape[1]}')
data_obj1.head()

Registros cargados: 7,964,177
Columnas: 13


,idinventariomes,idempresa,idsucursal,idalmacen,idproducto,inventariomes_fecha,inventariomes_revisada,inventariomesdetalle_diferencia,inventariomesdetalle_difimporte,inventariomesdetalle_costopromedio,almacen_nombre,producto_nombre,idcategoria
0,4207,3,9,53,65933,2017-10-01,1,48.0,2724.0,56.75,Almacén general,CINTURON DAMA,3
1,4209,3,9,53,65933,2017-10-08,1,0.0,0.0,56.75,Almacén general,CINTURON DAMA,3
2,4789,3,9,53,65933,2017-10-15,1,0.0,0.0,56.75,Almacén general,CINTURON DAMA,3
3,4790,3,9,53,65933,2017-10-22,1,0.0,0.0,56.75,Almacén general,CINTURON DAMA,3
4,4791,3,9,53,65933,2017-10-29,1,0.0,0.0,56.75,Almacén general,CINTURON DAMA,3


- De los ~11,000,000 de datos, al aplicar los filtros terminamos con 7,964,177 registros!

- Antes de calcular el z-score revisamos la distribución de diferencias para entender qué hay en los datos y verificar que los filtros aplicados en el preprocesamiento en SQL funcionaron.

In [3]:
total = len(data_obj1)

# Ver diferencias nulas. Estas no sirven para el Z-score
nulos_diff = data_obj1['inventariomesdetalle_diferencia'].isna().sum()

# Ver diferencias en cero. Estas indican que todo cuadr. No son anomalías pero si deben estar en el historial
ceros_diff = (data_obj1['inventariomesdetalle_diferencia'] == 0).sum()

# Ver diferencias negativas. Indican que hay menos producto del esperado
faltantes = (data_obj1['inventariomesdetalle_diferencia'] < 0).sum()

# Ver diferencias positivas. Indican que hay más producto del esperado
sobrantes = (data_obj1['inventariomesdetalle_diferencia'] > 0).sum()

print(f"Total registros: {total:,}")
print(f"\nDiferencia = NULL: {nulos_diff:,} ({nulos_diff/total*100:.1f}%)")
print(f"Diferencia = 0 (normal): {ceros_diff:,} ({ceros_diff/total*100:.1f}%)")
print(f"Diferencia < 0 (faltante): {faltantes:,} ({faltantes/total*100:.1f}%)")
print(f"Diferencia > 0 (sobrante): {sobrantes:,} ({sobrantes/total*100:.1f}%)")

Total registros: 7,964,177

Diferencia = NULL: 0 (0.0%)
Diferencia = 0 (normal): 2,424,976 (30.4%)
Diferencia < 0 (faltante): 2,850,735 (35.8%)
Diferencia > 0 (sobrante): 2,688,466 (33.8%)


- 30.4% de los registros tienen diferencia = 0 -> significa que en casi un tercio de los casos el inventario no sobra ni falta producto. 
- El 35.8% de los registros tiene faltantes y el 33.8% tiene sobrantes. El modelo detectará cuándo esos faltantes o sobrantes son inusualmente grandes comparados con el historial de ese producto.

#### 1. Calcular estadísticas históricas
- Para cada combinación única de idproducto + idalmacén calculamos media y desviación estándar histórica de sus diferencias.
- De esta forma vemos qué es normal para ese producto en ese almacén específico.

In [4]:
# Ordenar por producto, almacén y fecha para respetar el orden temporal
df_sorted = data_obj1.sort_values(['idproducto', 'idalmacen', 'inventariomes_fecha'])

# Expanding window: para cada fila, usa solo los cierres anteriores
expanding = df_sorted.groupby(['idproducto', 'idalmacen'])['inventariomesdetalle_diferencia'].expanding()

# shift(1) asegura que cada fila use estadísticas de cierres anteriores, no el propio
df_sorted['media_historica'] = expanding.mean().shift(1).values
df_sorted['std_historica'] = expanding.std().shift(1).values
df_sorted['n_cierres'] = expanding.count().shift(1).values

data_obj1 = df_sorted

- Hay 232,653 combinaciones distintas de producto + almacén en los datos. Estos son los datos que necesitamos para calcular el z-score d ecualquier cierre nuevo.

#### 2. Calculamos el Z-score 
- z = (diferencia_actual - media_histórica) / std_histórica

In [5]:
# Calcular Z-score: z = (diferencia_actual - media_histórica) / std_histórica
data_obj1['zscore'] = (data_obj1['inventariomesdetalle_diferencia'] - data_obj1['media_historica'])/data_obj1['std_historica']

### 3. Clasificamos por severidad:
- Usamos el valor absoluto del Z-score porque tanto faltantes (z negativo) como sobrantes (z positivo) son anomalías relevantes para el negocio.
- Sabemos que en una distribución normal:
    - El 68% de los datos cae dentro de |z| < 1
    - El 95% de los datos cae dentro de |z| < 2
    - El 99.7% de los datos cae dentro de |z| < 3
- Entonces clasificamos las alertas según los siguientes umbrales:
    - |z| < 2 -> el valor cae dentro del 95% más común (normal)
    - |z| 2-3 -> el valor está en el 5%  más extremo (alerta)
    - |z| > 3 -> el valor está en el 0.3% más extremo (alerta crítica)

In [6]:
# Función para clasificar la alerta de cada registro según los umbrales definidos para el z-score
def clasificar_severidad(z): 
    """
    Clasifica la severidad de una anomalía de inventario basándose en el valor del z-score.
    
    Args:
        z (float): Z-score calculado para un producto en un cierre específico.

    Returns:
        string: Etiqueta de severidad:
    """
    if pd.isna(z):
        return 'sin_historial' # z es nulo, std es 0 o no hay historial
    elif abs(z) < 2:
        return 'normal'
    elif abs(z) < 3:
        return 'alerta'
    else:
        return 'critico'
    
# Aplicar funcion al z-score calculado a los datos
data_obj1['severidad'] = data_obj1['zscore'].apply(clasificar_severidad)

# Contar cuántos registros cayeron en cada categoría
print('Distribucion de severidad:')
conteo = data_obj1['severidad'].value_counts()
for sev, n in conteo.items():
    print(f'{sev}: {n:,} ({n/len(data_obj1)*100:.1f}%)')


Distribucion de severidad:
normal: 6,874,028 (86.3%)
sin_historial: 521,863 (6.6%)
critico: 305,151 (3.8%)
alerta: 263,135 (3.3%)


#### 4. Tabla de hallazgos 
- El z-score ya se calculó para todos los 7.9 millodes de registros. 
- Para simular cómo vería los hallazgos el auditor en la práctica, filtramos el cierre más reciente en los datos y generamos la tabla de hallazgos priorizada por impacto económico.
- Solo mostramos alertas y críticos ya que los normales no requieren atención del auditor.

In [7]:
# Tabla de hallazgos del cierre más reciente

# Tomamos el cierre más reciente
cierre_reciente = data_obj1['idinventariomes'].max() # idinventariomes crece con el tiempo, el mas alto corresponde al cierre más reciente.
fecha_cierre = data_obj1[data_obj1['idinventariomes'] == cierre_reciente]['inventariomes_fecha'].iloc[0] # tomamos la primera fecha de ese cierre

# Imprimir ID y fecha del cierre a analizar
print(f"Cierre analizado: {cierre_reciente} — {fecha_cierre.date()}")

hallazgos = (
    # Filtrar solo anomalias del cierre reciente
    data_obj1[
    (data_obj1['idinventariomes'] == cierre_reciente) &  # solo filas del cierre mas reciente
    (data_obj1['severidad'].isin(['alerta', 'critico'])) # solo filas donde la severidad sea alerta o critico
    ]
    # Ordenar por impacto económico. Ordenamos por difimporte, en valor absoluto, de mayor a menor.
    .sort_values('inventariomesdetalle_difimporte', key = abs, ascending = False)
    # Seleccionar solo las columnas relevantes para la tabla de hallazgos
    [['severidad',
      'producto_nombre',
      'almacen_nombre',
      'inventariomesdetalle_diferencia',
      'inventariomesdetalle_difimporte',
      'zscore'
      ]]
    # Renombramos columnas de la tabla para que sean mas legibles
    .rename(columns = {
        'severidad': 'Severidad',
        'producto_nombre': 'Producto',
        'almacen_nombre': 'Almacén',
        'inventariomesdetalle_diferencia': 'Diferencia',
        'inventariomesdetalle_difimporte': 'Impacto ($)',
        'zscore': 'Z-score'
    }).head(20).reset_index(drop = True) # limitamos resultados ya que estan ordenados por impacto, y reiniciamos indice del dataframe
    )

# Imprimimos el resultado!
print(f"\nHallazgos encontrados: {len(hallazgos)}")
hallazgos.round(2)

Cierre analizado: 118889 — 2026-04-12

Hallazgos encontrados: 7


,Severidad,Producto,Almacén,Diferencia,Impacto ($),Z-score
0,alerta,VINO TINTO CALIFORNIA / DON SIMON LT,Barra,26.00,1577.94,2.43
1,alerta,CARLO ROSSI LT,Barra,-20.92,-1269.63,-2.79
2,alerta,HUITZILA LT,Barra,4.50,517.77,2.19
3,critico,JUGO DE ARANDANO LT,Barra,5.95,490.88,4.58
4,critico,JUGO DE MANZANA CLARIFICADA LT,Barra,-1.00,-30.00,-7.74
5,alerta,ZARZAMORA KG,Barra,-0.13,-27.53,-2.62
6,alerta,FRAMBUESA KG,Barra,-0.13,-27.30,-2.62


- La idea es que el copiloto responda "dado el cierre de esta semana, ¿Qué productos debo revisar primero?
- Cada semana cuando el auditor abra el copiloto, selecciona su cierre y vería solo los hallazgos de esa semana.

### 5. Análisis Histórico

#### 5.1 Anomalias por cierre a lo largo del tiempo
- Ahora, en lugar de ver solo un cierre específico, aquí analizamos todos los cierres históricos para entender el comportamiento general del sistema. 
- Esto nos permite responder:
    - ¿Cuántas anomalías detecta el modelo por cierre en promedio?
    - ¿Hay cierres con impacto económico especialmente alto?

In [ ]:
# Análisis histórico: Anomalías por cierre

anomalias_por_cierre = (
    # Filtramos solo los registros marcados como alerta o crítico
    data_obj1[data_obj1['severidad'].isin(['alerta', 'critico'])]
    
    # Agrupamos por cierre, cada idinventariomes es un cierre de semana de un almacén específico. 
    # Incluimos la fecha para poder graficar la evolución temporal después.
    .groupby(['idinventariomes', 'inventariomes_fecha'])
    
    # Para cada cierre calculamos 4 métricas:
    .agg(
        # 1. Cuántos productos anómalos (alerta o crítico) tiene ese cierre
        total_anomalias = ('severidad', 'count'),
        
        # 2. Suma del valor absoluto del impacto económico, ósea cuánto dinero está en juego en ese cierre entre todas sus anomalías
        # Usamos abs() porque faltantes = negativos y sobrantes = positivos, pero ambos representan dinero en riesgo
        impacto_total = ('inventariomesdetalle_difimporte', 
                         lambda x: x.abs().sum()),
        
        # 3. Cuántos de esos productos anómalos son críticos (|z| > 3)
        criticos = ('severidad', lambda x: (x == 'critico').sum()),
        
        # 4. Cuántos son solo alerta (2 < |z| < 3)
        alertas = ('severidad', lambda x: (x == 'alerta').sum())
    )
    .reset_index()
    
    # Ordenamos por fecha para ver la evolución cronológica
    .sort_values('inventariomes_fecha')
)

In [ ]:
print(f"Total de cierres con anomalías: {anomalias_por_cierre.shape[0]:,}")

print(f"\nEstadísticas de anomalías por cierre:")
print(anomalias_por_cierre[[
    'total_anomalias', 
    'impacto_total', 
    'criticos', 
    'alertas'
]].describe().round(2))

Total de cierres con anomalías: 38,888

Estadísticas de anomalías por cierre:
       total_anomalias  impacto_total  criticos   alertas
count         38888.00   3.888800e+04  38888.00  38888.00
mean             10.18   1.339326e+08      3.95      6.23
std              17.38   2.189511e+10     10.69      8.88
min               1.00   0.000000e+00      0.00      0.00
25%               2.00   1.217520e+03      0.00      1.00
50%               5.00   5.722540e+03      1.00      3.00
75%              11.00   2.336386e+04      4.00      8.00
max             490.00   4.264667e+12    420.00    266.00


#### 5.2 Top productos más problemáticos históricamente

- Aquí identificamos qué productos han generado mñas anomalías y mayor impacto económico a lo laro de toooda la histori de los datos. 
- Respondemos:
    - ¿Qué productos son sistemáticamente problemáticos?
    - ¿Cuánto dinero acumulado representan esas anomalías?
- Identificamos los productos que rquieren atención estructural, no solo en un cierre puntual, sino de forma rcurrente.

In [ ]:
# Análisis histórico: Top productos más problemáticos

top_productos = (
    # Solo anomalías (alerta y crítico)
    data_obj1[data_obj1['severidad'].isin(['alerta', 'critico'])]
    
    # Agrupamos por producto ya que queremos ver el historial completo de cada producto en todos los cierres y almacenes
    .groupby('producto_nombre')
    
    # Para cada producto calculamos 3 métricas históricas:
    .agg(
        # 1. Cuántas veces apareció como anómalo en toda la historia 
        veces_anomalo = ('severidad', 'count'),
        
        # 2. Suma del impacto económico acumulado en todos los cierres
        # Esto nos dice cuánto dinero total representa ese producto
        # como problema a lo largo del tiempo
        impacto_acumulado = ('inventariomesdetalle_difimporte', 'sum'),
        
        # 3. Z-score promedio, ósea, qué tan anómalas son sus diferencias en promedio. 
        # Un Z-score promedio muy alto indica que ese producto tiene desviaciones muy grandes habitualmente
        zscore_promedio = ('zscore', 'mean')
    )
    .reset_index()
    
    # Ordenamos por impacto acumulado en valor absoluto porque nos interesan los productos con mayor impacto económico total, ya sean faltantes o sobrantes
    .sort_values('impacto_acumulado', key = abs, ascending=False)
    
    # Solo los 20 más problemáticos
    .head(20)
)

print("Top 20 productos más problemáticos históricamente:")
top_productos.round(2)

Top 20 productos más problemáticos históricamente:


,producto_nombre,veces_anomalo,impacto_acumulado,zscore_promedio
8606,CASA MADERO MERLOT 750 ML,16,3.880847e+12,-1.01
15928,FILETE IMPORTADO 600 GRS,3,-5.552200e+11,-3.51
28330,"PAPEL PARA ALIMENTO COTIDIANO PAQ 1,000PZ",20,-1.432236e+09,-1.49
35527,Servilleta Advanced,39,3.869512e+08,-0.67
5784,BLOCK COMANDA,32,-1.270834e+08,-2.26
3505,ALGA NORI PZA,14,-5.614409e+07,-1.46
20723,JUGO DE LIMON VERDE GAL,7,5.060793e+07,-1.81
28326,PAPEL ORRACA PLIEGO,21,-2.535034e+07,0.18
21149,KING CRAB KG,27,2.257610e+07,0.02
6368,BONELESS Porcion 130 Grs,21,-1.626368e+07,0.18
